In [4]:
import MDAnalysis as mda
import numpy as np

u = mda.Universe("/home/leandro/mcu_emre/nvt6.gro")

print("Caja:", u.dimensions)

for resname in ["POPC", "POPE", "POPS", "CHL1"]:
    ag = u.select_atoms(f"resname {resname}")
    if len(ag):
        z = ag.positions[:,2]
        print(f"{resname:5s} N={len(ag):7d}  Zmin={z.min():8.3f}  Zmax={z.max():8.3f}  Zmean={z.mean():8.3f}")

Caja: [152.496   152.496   331.48572  90.       90.       90.     ]
POPC  N=  76380  Zmin=  38.660  Zmax= 266.740  Zmean= 143.608
POPE  N=  55750  Zmin=  41.830  Zmax= 265.410  Zmean= 145.871
POPS  N=  41910  Zmin=  41.560  Zmax= 265.990  Zmean= 146.321
CHL1  N=  15392  Zmin=  48.430  Zmax= 257.350  Zmean= 153.327


## Identificar el agua

In [1]:
import MDAnalysis as mda
from collections import Counter

u = mda.Universe("/home/leandro/mcu_emre/nvt6.gro")

print(Counter(u.atoms.resnames).most_common(30))

[('TIP3', 576849), ('POPC', 76380), ('POPE', 55750), ('POPS', 41910), ('CHL1', 15392), ('LEU', 3724), ('ARG', 2976), ('VAL', 1984), ('LYS', 1672), ('ILE', 1596), ('ASP', 1356), ('GLU', 1320), ('TYR', 1260), ('CLA', 1236), ('ALA', 1120), ('PHE', 1120), ('SER', 1100), ('GLN', 1088), ('PRO', 1008), ('THR', 952), ('TRP', 576), ('GLY', 564), ('MET', 552), ('CAL', 530), ('POT', 506), ('HSD', 408), ('ASN', 392), ('CYS', 88)]


In [2]:
import MDAnalysis as mda
import numpy as np

gro = "/home/leandro/mcu_emre/nvt6.gro"

u = mda.Universe(gro)

water = u.select_atoms("resname TIP3 and name OH2")

z = water.positions[:, 2]

bins = np.arange(0, u.dimensions[2] + 1, 1.0)

hist, edges = np.histogram(z, bins=bins)

centers = 0.5 * (edges[:-1] + edges[1:])

for zc, n in zip(centers, hist):
    print(f"{zc:8.2f} {n:8d}")

    0.50      766
    1.50      767
    2.50      780
    3.50      761
    4.50      759
    5.50      797
    6.50      765
    7.50      808
    8.50      770
    9.50      763
   10.50      809
   11.50      781
   12.50      759
   13.50      815
   14.50      801
   15.50      753
   16.50      800
   17.50      786
   18.50      810
   19.50      706
   20.50      841
   21.50      772
   22.50      767
   23.50      808
   24.50      735
   25.50      807
   26.50      795
   27.50      767
   28.50      750
   29.50      794
   30.50      773
   31.50      786
   32.50      776
   33.50      760
   34.50      763
   35.50      798
   36.50      750
   37.50      784
   38.50      807
   39.50      736
   40.50      816
   41.50      805
   42.50      740
   43.50      783
   44.50      720
   45.50      678
   46.50      618
   47.50      590
   48.50      495
   49.50      462
   50.50      356
   51.50      331
   52.50      264
   53.50      202
   54.50      133
   55.50  

In [3]:
import MDAnalysis as mda
from collections import Counter

u = mda.Universe("/home/leandro/mcu_emre/nvt6.gro")

print("RESIDUOS MÁS ABUNDANTES:")
for res, n in Counter(u.atoms.resnames).most_common(30):
    print(f"{res:10s} {n:10d}")

print("\nÁTOMOS DE AGUA CANDIDATOS:")
for name in ["OH2", "OW", "O", "OT"]:
    sel = u.select_atoms(f"name {name}")
    print(f"{name:5s}: {len(sel)}")

RESIDUOS MÁS ABUNDANTES:
TIP3           576849
POPC            76380
POPE            55750
POPS            41910
CHL1            15392
LEU              3724
ARG              2976
VAL              1984
LYS              1672
ILE              1596
ASP              1356
GLU              1320
TYR              1260
CLA              1236
ALA              1120
PHE              1120
SER              1100
GLN              1088
PRO              1008
THR               952
TRP               576
GLY               564
MET               552
CAL               530
POT               506
HSD               408
ASN               392
CYS                88

ÁTOMOS DE AGUA CANDIDATOS:
OH2  : 192283
OW   : 0
O    : 1524
OT   : 0


In [4]:
import MDAnalysis as mda
import numpy as np

gro = "/home/leandro/mcu_emre/nvt6.gro"

u = mda.Universe(gro)

# ============================================================
# 1. Seleccionar oxígenos del agua
# ============================================================

water_O = u.select_atoms("resname TIP3 and name OH2")

print(f"Átomos de oxígeno de agua: {len(water_O)}")

# ============================================================
# 2. Histograma de agua a lo largo de Z
# ============================================================

z = water_O.positions[:, 2]

dz = 1.0  # Å
zmax = u.dimensions[2]

bins = np.arange(0, zmax + dz, dz)

hist, edges = np.histogram(z, bins=bins)

centers = 0.5 * (edges[:-1] + edges[1:])

# ============================================================
# 3. Guardar perfil
# ============================================================

outfile = "water_density_Z.dat"

with open(outfile, "w") as f:
    f.write("# Z_Angstrom    N_water_O\n")

    for zc, n in zip(centers, hist):
        f.write(f"{zc:10.3f} {n:10d}\n")

print(f"\nPerfil guardado en: {outfile}")

# ============================================================
# 4. Mostrar regiones con mayor densidad
# ============================================================

print("\nValores del perfil cada 5 Å:\n")

for i in range(0, len(centers), 5):
    print(f"{centers[i]:8.1f} Å   {hist[i]:6d}")

Átomos de oxígeno de agua: 192283

Perfil guardado en: water_density_Z.dat

Valores del perfil cada 5 Å:

     0.5 Å      766
     5.5 Å      797
    10.5 Å      809
    15.5 Å      753
    20.5 Å      841
    25.5 Å      807
    30.5 Å      773
    35.5 Å      798
    40.5 Å      816
    45.5 Å      678
    50.5 Å      356
    55.5 Å       92
    60.5 Å        3
    65.5 Å        0
    70.5 Å        0
    75.5 Å        0
    80.5 Å        2
    85.5 Å       30
    90.5 Å      294
    95.5 Å      632
   100.5 Å      767
   105.5 Å      773
   110.5 Å      772
   115.5 Å      773
   120.5 Å      765
   125.5 Å      757
   130.5 Å      743
   135.5 Å      736
   140.5 Å      749
   145.5 Å      777
   150.5 Å      786
   155.5 Å      735
   160.5 Å      796
   165.5 Å      786
   170.5 Å      815
   175.5 Å      789
   180.5 Å      781
   185.5 Å      795
   190.5 Å      803
   195.5 Å      773
   200.5 Å      749
   205.5 Å      795
   210.5 Å      703
   215.5 Å      431
   220.5 Å    